# Context-Conditioned Album Cover Generation — Experiments

Trains four context conditions on the same audio/cover pairs and produces a
qualitative comparison grid per held-out song.

Conditions:
1. `audio_only` — LoRA fine-tune, audio embedding only (no artist text)
2. `style`     — audio + artist style description
3. `biography` — audio + artist biography
4. `combined`  — audio + biography + style

Plus a no-training baseline:
5. `caption`   — pretrained SD prompted with the artist text only, no audio

Final paper figures land under `paper/figures/`.

## 1. GPU + Drive

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Mickey_Mouse

## 2. Install deps

In [ ]:
!pip install -q -r requirements.txt
# Colab ships torchao 0.10.0; the current peft requires >=0.16.0 and refuses
# to import otherwise. We don't use torchao at all, so just uninstall it.
!pip uninstall -y torchao

## 3. Build the CLAP+VAE cache (once)

Reads `arianasongs/*.mp3`, `arianacovers/*.{jpg,png}` etc. and writes
`cache/<artist>__<stem>.pt`. About 5–10 minutes first time.

In [ ]:
!python prepare_data.py --cache-dir cache

## 4. Train all four context modes

Each mode trains a fresh LoRA+projector for `EPOCHS` epochs on the same
cached pairs and is saved under `checkpoints/<mode>/ckpt_eXXXX/`.
With 60 pairs and batch 4: ~15 steps/epoch. 50 epochs ≈ 750 steps ≈ ~25 min
per mode on a T4, ~1.5–2 hours total. Bump `EPOCHS=100` if loss is still
dropping at the end of run 1.

The shell script is idempotent within a mode: re-running just re-creates
the same checkpoint dir.

In [ ]:
!EPOCHS=50 bash run_context_experiments.sh

## 5. Caption-only baseline (no training)

Asks pretrained SD 1.5 to generate covers using only the artist text — no
audio in the loop at all. Same seeds as the trained eval so the rows line up.

In [ ]:
!python caption_baseline.py --mode combined --seed 100 --output-dir outputs/heldout_varied_seeds

## 6. Held-out evaluation

Generates an image per (song, trained_mode) pair and assembles per-song
comparison sheets that include GT, the caption baseline, and the four
trained modes.

In [ ]:
!python evaluate_heldout.py --seed 100 --output-dir outputs/heldout_varied_seeds

## 7. Build the paper figures

In [ ]:
!python make_paper_figures.py --input-dir outputs/heldout_varied_seeds --output-dir paper/figures

## 8. Peek at one

In [ ]:
from PIL import Image
Image.open('paper/figures/fig_grid_drake.png')